# Module 3 — Prompt Engineering & Cost-Aware Design

**Hands-on objective:** Turn prompting into an engineering discipline using progressive prompt control, few-shot patterns, structured outputs, prompt chaining, context reduction, token economics, and a reusable Healthcare Policy Assistant framework.

> “Module 2 showed us how models behave. Module 3 shows us how to deliberately control that behaviour.”


## Secure configuration loading

**What this demonstrates:** Load endpoint, credential, and deployment settings from the `.env` file instead of hard-coding secrets in the notebook.

**What to observe:** The notebook should confirm that configuration values exist without exposing the API key itself.


In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env", override=True)

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
model_primary = os.getenv("AZURE_OPENAI_MODEL")

client = OpenAI(
    base_url=endpoint,
    api_key=api_key
)

print("Azure client ready")
print("Primary model:", model_primary)

Azure client ready
Primary model: gpt-4.1-mini


## First LLM interaction

**What this demonstrates:** Send a healthcare payer question to the deployed Azure model and retrieve the generated response.

**What to observe:** Observe how a simple user message becomes a structured API request and returns an assistant message.


In [2]:
policy_question = "Explain when prior authorization is required for MRI procedures."

prompt_versions = {
    "Basic": policy_question,

    "Role + Context": """
You are a healthcare payer policy assistant.
Explain when prior authorization is required for MRI procedures
for a healthcare technology professional.
""",

    "Controlled": """
You are a healthcare payer policy assistant.

Task:
Explain when prior authorization is required for MRI procedures.

Requirements:
- Use payer terminology.
- Keep the answer concise.
- Maximum 3 bullet points.
- Do not invent policy details that are not provided.
"""
}

responses = {}

for name, prompt in prompt_versions.items():

    response = client.chat.completions.create(
        model=model_primary,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    responses[name] = response

    print(f"\n{name.upper()}")
    print("-" * 60)
    print(response.choices[0].message.content)


BASIC
------------------------------------------------------------
Prior authorization for MRI procedures is typically required by health insurance providers to ensure that the imaging is medically necessary and appropriate for the patient's condition before the service is performed. The specifics can vary by insurance plan and region, but generally, prior authorization for MRI is required in the following situations:

1. **Non-Emergent Imaging**: Most insurers require prior authorization for MRI scans that are planned in advance and are not related to an emergency or urgent care situation.

2. **Certain Anatomical Sites**: Some insurance policies specify prior authorization requirements based on the body part being imaged. For example, MRIs of the spine, brain, or joints may have distinct criteria that need review.

3. **High-Cost or Advanced Imaging**: Because MRIs are relatively expensive diagnostic tests, prior authorization helps control costs and prevent unnecessary procedures.


## First LLM interaction

**What this demonstrates:** Send a healthcare payer question to the deployed Azure model and retrieve the generated response.

**What to observe:** Observe how a simple user message becomes a structured API request and returns an assistant message.


In [3]:
policy_question = "Explain when prior authorization is required for MRI procedures."

prompt_versions = {
    "Basic": policy_question,

    "Role + Context": """
You are a healthcare payer policy assistant.
Explain when prior authorization is required for MRI procedures
for a healthcare technology professional.
""",

    "Controlled": """
You are a healthcare payer policy assistant.

Task:
Explain when prior authorization is required for MRI procedures.

Requirements:
- Use payer terminology.
- Keep the answer concise.
- Maximum 3 bullet points.
- Do not invent policy details that are not provided.
"""
}

responses = {}

for name, prompt in prompt_versions.items():

    response = client.chat.completions.create(
        model=model_primary,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    responses[name] = response

    print(f"\n{name.upper()}")
    print("-" * 60)
    print(response.choices[0].message.content)


BASIC
------------------------------------------------------------
Prior authorization for MRI (Magnetic Resonance Imaging) procedures is typically required by health insurance plans to ensure that the imaging is medically necessary and appropriate before the service is performed. While specific policies can vary by insurer and plan, prior authorization is generally required in the following situations:

1. **Non-Emergency (Elective) MRIs:** Most health plans require prior authorization for scheduled, non-emergency MRI scans to confirm that the imaging is justified based on symptoms, clinical indications, or prior treatments.

2. **Outpatient or Ambulatory Settings:** MRI procedures performed in outpatient clinics, imaging centers, or ambulatory care settings often need prior approval, as insurers scrutinize cost-effective care.

3. **Certain Body Areas or Conditions:** Some insurers have specific criteria for MRIs of certain regions (e.g., spine, brain, joints) or conditions to preve

## Few-shot + structured output

**What this demonstrates:** Combine examples with a target schema so the model learns both the content pattern and machine-readable response shape.

**What to observe:** The output should follow the demonstrated JSON structure and stay grounded in the supplied policy context.


In [4]:
import json

few_shot_prompt = """
You are a healthcare payer policy assistant.

Return the answer as JSON with these fields:
- policy_topic
- authorization_required
- reason
- confidence

Examples:

Question: Does physical therapy require authorization after 10 visits?
Answer:
{
  "policy_topic": "Physical Therapy",
  "authorization_required": true,
  "reason": "Authorization is required after the initial 10 visits.",
  "confidence": "high"
}

Question: Is prior authorization required for emergency imaging?
Answer:
{
  "policy_topic": "Emergency Imaging",
  "authorization_required": false,
  "reason": "The provided policy states emergency imaging is exempt.",
  "confidence": "high"
}

Now answer:

Question: Does an MRI require prior authorization?

Policy context:
MRI procedures require prior authorization.
"""

response_structured = client.chat.completions.create(
    model=model_primary,
    messages=[
        {
            "role": "user",
            "content": few_shot_prompt
        }
    ]
)

print(response_structured.choices[0].message.content)

{
  "policy_topic": "MRI",
  "authorization_required": true,
  "reason": "MRI procedures require prior authorization according to the policy.",
  "confidence": "high"
}


**Takeaway:** Few-shot examples teach the pattern; structured output makes the answer usable by software.


## First LLM interaction

**What this demonstrates:** Send a healthcare payer question to the deployed Azure model and retrieve the generated response.

**What to observe:** Observe how a simple user message becomes a structured API request and returns an assistant message.


In [5]:
# Step 1: Extract a structured policy decision

policy_context = """
MRI procedures require prior authorization.
Emergency imaging is exempt from prior authorization.
"""

user_question = "The provider wants to schedule a non-emergency MRI. What should happen?"

extract_response = client.chat.completions.create(
    model=model_primary,
    messages=[
        {
            "role": "user",
            "content": f"""
Using only the policy below, extract the decision.

Policy:
{policy_context}

Question:
{user_question}

Return only:
Authorization Required: Yes/No
Reason:
"""
        }
    ]
)

policy_decision = extract_response.choices[0].message.content

print("STEP 1 - POLICY DECISION")
print(policy_decision)

STEP 1 - POLICY DECISION
Authorization Required: Yes  
Reason: MRI procedures require prior authorization unless they are emergency imaging, which is exempt. Since the MRI is non-emergency, prior authorization is required.


## Prompt chain — operational action

**What this demonstrates:** Feed the first model's decision into a second prompt that recommends the next operational step.

**What to observe:** The second step should preserve the policy decision rather than reinterpret it.


In [6]:
action_response = client.chat.completions.create(
    model=model_primary,
    messages=[
        {
            "role": "system",
            "content": """
You are a healthcare operations assistant.
Use the policy decision provided to recommend the next operational action.
Do not change or reinterpret the policy decision.
Keep the response to one concise sentence.
"""
        },
        {
            "role": "user",
            "content": f"""
Policy decision:
{policy_decision}

What should the operations team do next?
"""
        }
    ]
)

print("STEP 2 - OPERATIONAL ACTION")
print(action_response.choices[0].message.content)

STEP 2 - OPERATIONAL ACTION
The operations team should initiate the prior authorization process for the non-emergency MRI procedure.


**Takeaway:** A chain can separate 'What does the policy say?' from 'What should the workflow do?' — exactly how enterprise orchestration becomes controllable.


## Context optimization setup

**What this demonstrates:** Create a full policy context and a reduced context containing only the information relevant to the MRI question.

**What to observe:** Both contexts support the same answer, but one carries substantial irrelevant material.


In [7]:

full_context = """
MRI procedures require prior authorization.
Emergency imaging is exempt from prior authorization.
CT scans require prior authorization for outpatient procedures.
Physical therapy requires authorization after 10 visits.
Specialist consultations do not require prior authorization.
Dental coverage is excluded from this plan.
Vision coverage is available once every 24 months.
Member address changes must be updated within 30 days.
"""

reduced_context = """
MRI procedures require prior authorization.
Emergency imaging is exempt from prior authorization.
"""

question = "Does a non-emergency MRI require prior authorization?"

**Takeaway:** The cheapest token is the one you never send.


## Context reduction + token economics

**What this demonstrates:** Run the same question with full and reduced context and compare prompt-token consumption.

**What to observe:** If the answer remains unchanged while tokens drop sharply, the reduced context is more efficient.


In [8]:
full_response = client.chat.completions.create(
    model=model_primary,
    messages=[
        {
            "role": "system",
            "content": "Answer only from the provided policy context."
        },
        {
            "role": "user",
            "content": f"""
Context:
{full_context}

Question:
{question}
"""
        }
    ]
)

reduced_response = client.chat.completions.create(
    model=model_primary,
    messages=[
        {
            "role": "system",
            "content": "Answer only from the provided policy context."
        },
        {
            "role": "user",
            "content": f"""
Context:
{reduced_context}

Question:
{question}
"""
        }
    ]
)

print("FULL CONTEXT")
print(full_response.choices[0].message.content)
print("Prompt tokens:", full_response.usage.prompt_tokens)

print("\nREDUCED CONTEXT")
print(reduced_response.choices[0].message.content)
print("Prompt tokens:", reduced_response.usage.prompt_tokens)

FULL CONTEXT
Yes, a non-emergency MRI requires prior authorization.
Prompt tokens: 104

REDUCED CONTEXT
Yes, a non-emergency MRI requires prior authorization.
Prompt tokens: 48


**Takeaway:** In enterprise RAG, retrieval quality is a cost-control mechanism as much as a relevance mechanism.


## Token usage inspection

**What this demonstrates:** Measure how many tokens were consumed by the input prompt and generated completion.

**What to observe:** Prompt, completion, and total token counts are the basic unit for understanding model consumption.


In [9]:
def healthcare_policy_assistant(question, policy_context):
    response = client.chat.completions.create(
        model=model_primary,
        messages=[
            {
                "role": "system",
                "content": """
You are a healthcare payer policy assistant.

Rules:
1. Use only the provided policy context.
2. Do not invent missing policy details.
3. If the evidence is insufficient, say "Insufficient information".
4. Keep the response concise and operational.
5. Return the answer in this structure:

Decision:
Reason:
Next Action:
"""
            },
            {
                "role": "user",
                "content": f"""
Policy Context:
{policy_context}

Question:
{question}
"""
            }
        ]
    )

    return {
        "answer": response.choices[0].message.content,
        "prompt_tokens": response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens,
        "total_tokens": response.usage.total_tokens
    }

**Takeaway:** Every extra instruction, document chunk, or generated paragraph has a measurable cost footprint — AI architecture is also token architecture.


## Hands-on experiment

**What this demonstrates:** Execute the cell and connect the code behaviour to the current AI engineering concept.

**What to observe:** Focus on the output structure, model behaviour, and measurable metadata.


In [10]:
policy_context = """
MRI procedures require prior authorization.
Emergency imaging is exempt from prior authorization.
Physical therapy requires authorization after 10 visits.
"""

question = "A provider wants to schedule a non-emergency MRI. What should happen?"

result = healthcare_policy_assistant(
    question=question,
    policy_context=policy_context
)

print(result["answer"])

print("\nTOKEN USAGE")
print("Prompt tokens     :", result["prompt_tokens"])
print("Completion tokens :", result["completion_tokens"])
print("Total tokens      :", result["total_tokens"])

Decision:
The non-emergency MRI requires prior authorization.

Reason:
MRI procedures require prior authorization unless it is an emergency, which this is not.

Next Action:
The provider should submit a prior authorization request before scheduling the MRI.

TOKEN USAGE
Prompt tokens     : 123
Completion tokens : 46
Total tokens      : 169
